# Notebook 02: Feature Engineering

**Purpose**: Compute daily behavioral metrics for each animal from the cleaned drinking visit data. These metrics are the features that the anomaly detection model will use.

**Input**: `scanner_data_clean.csv` (32,825 drinking visits), `animals.csv` (34 animals)  
**Output**: `daily_features.csv` — one row per animal per day, with 4 behavioral metrics

**Metrics** (as specified in the thesis methodology):
1. Total daily drinking duration
2. Frequency of visits
3. Average duration per visit
4. Maximum absence from water

In [1]:
import pandas as pd
import numpy as np
import os

# Paths
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')

# Load cleaned data from Phase 1
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'scanner_data_clean.csv'), parse_dates=['start_dt', 'end_dt'])
df['date'] = pd.to_datetime(df['date'])
df_animals = pd.read_csv(os.path.join(PROCESSED_DIR, 'animals.csv'))

print(f"Loaded: {len(df):,} drinking visits, {df['tag_short'].nunique()} animals")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

Loaded: 32,118 drinking visits, 30 animals
Date range: 2024-10-11 to 2024-11-07


,tag_id,tag_short,antenna,start_dt,end_dt,duration_sec,date,device_id
0,999999999999000000001101,1101,1,2024-10-11 05:04:18.175,2024-10-11 05:10:33.041,374.866,2024-10-11,d4a7af59e92892fb
1,999999999999000000001101,1101,1,2024-10-11 07:00:01.059,2024-10-11 07:00:01.295,0.236,2024-10-11,d4a7af59e92892fb
2,999999999999000000001101,1101,1,2024-10-11 07:00:03.613,2024-10-11 07:00:07.074,3.461,2024-10-11,d4a7af59e92892fb
3,999999999999000000001101,1101,1,2024-10-11 07:00:09.381,2024-10-11 07:00:10.863,1.482,2024-10-11,d4a7af59e92892fb
4,999999999999000000001101,1101,1,2024-10-11 07:00:20.714,2024-10-11 07:00:24.424,3.710,2024-10-11,d4a7af59e92892fb


## Step 1: Merge Scanner Sessions into Drinking Bouts

The RFID scanner creates multiple short sessions for what is actually one continuous drinking bout. Analysis of inter-session gaps shows that **53% of gaps are under 5 seconds** and **80% are under 30 seconds** — these are not separate trips to the drinker, but brief signal interruptions during a single visit.

We merge consecutive sessions separated by **less than 60 seconds** into a single **drinking bout**. This produces physiologically meaningful "visits" (true trips to the water trough) rather than raw scanner artifacts.

In [2]:
# Sort by animal and time
df = df.sort_values(['tag_short', 'start_dt']).reset_index(drop=True)

# Compute gap to previous session (same animal)
df['prev_end'] = df.groupby('tag_short')['end_dt'].shift(1)
df['gap_sec'] = (df['start_dt'] - df['prev_end']).dt.total_seconds()

# Show gap distribution to justify the 60-second threshold
gaps = df['gap_sec'].dropna()
print("Inter-session gap distribution:")
for p in [25, 50, 75, 90, 95]:
    print(f"  {p}th percentile: {gaps.quantile(p/100):.1f}s")
print(f"  Gaps < 60s: {(gaps < 60).sum()} ({(gaps < 60).mean()*100:.1f}%)")
print(f"  Gaps >= 60s: {(gaps >= 60).sum()} ({(gaps >= 60).mean()*100:.1f}%)")

# ---- Merge sessions into drinking bouts ----
BOUT_GAP_THRESHOLD = 60  # seconds

# A new bout starts when gap > threshold or it's the animal's first session
df['new_bout'] = (df['gap_sec'] > BOUT_GAP_THRESHOLD) | (df['gap_sec'].isna())
df['bout_id'] = df.groupby('tag_short')['new_bout'].cumsum()

# Aggregate sessions into bouts
bouts = df.groupby(['tag_short', 'bout_id']).agg(
    start_dt=('start_dt', 'min'),
    end_dt=('end_dt', 'max'),
    duration_sec=('duration_sec', 'sum'),  # total scanner-detected time within bout
    n_sessions=('duration_sec', 'count'),
    date=('date', 'first')
).reset_index()

print(f"\nSession merging (gap threshold = {BOUT_GAP_THRESHOLD}s):")
print(f"  Raw sessions: {len(df):,}")
print(f"  Merged bouts: {len(bouts):,}")
print(f"  Compression ratio: {len(df)/len(bouts):.1f}x")
print(f"\nBout statistics:")
print(f"  Mean sessions per bout: {bouts['n_sessions'].mean():.1f}")
print(f"  Mean bout duration: {bouts['duration_sec'].mean():.1f}s ({bouts['duration_sec'].mean()/60:.1f} min)")
print(f"  Median bout duration: {bouts['duration_sec'].median():.1f}s")
print(f"  Max bout duration: {bouts['duration_sec'].max():.1f}s ({bouts['duration_sec'].max()/60:.1f} min)")

Inter-session gap distribution:
  25th percentile: 2.2s
  50th percentile: 4.5s
  75th percentile: 17.5s
  90th percentile: 1788.5s
  95th percentile: 13058.8s
  Gaps < 60s: 26682 (83.2%)
  Gaps >= 60s: 5406 (16.8%)

Session merging (gap threshold = 60s):
  Raw sessions: 32,118
  Merged bouts: 5,436
  Compression ratio: 5.9x

Bout statistics:
  Mean sessions per bout: 5.9
  Mean bout duration: 60.2s (1.0 min)
  Median bout duration: 33.1s
  Max bout duration: 1101.2s (18.4 min)


In [3]:
# ---- Compute the 4 daily metrics from drinking bouts ----

# Sort bouts for inter-bout gap calculation
bouts = bouts.sort_values(['tag_short', 'start_dt']).reset_index(drop=True)

# Compute inter-bout gaps (for max_absence metric)
bouts['prev_bout_end'] = bouts.groupby('tag_short')['end_dt'].shift(1)
bouts['bout_gap_sec'] = (bouts['start_dt'] - bouts['prev_bout_end']).dt.total_seconds()

daily_features = bouts.groupby(['tag_short', 'date']).agg(
    total_daily_duration=('duration_sec', 'sum'),       # Metric 1: total seconds at drinker
    visit_count=('duration_sec', 'count'),               # Metric 2: number of drinking bouts
    avg_visit_duration=('duration_sec', 'mean'),         # Metric 3: mean bout duration
    max_absence_sec=('bout_gap_sec', 'max')              # Metric 4: longest inter-bout gap (seconds)
).reset_index()

# Convert max absence to hours
daily_features['max_absence_hours'] = daily_features['max_absence_sec'] / 3600

print(f"Feature matrix: {len(daily_features)} rows (animal-days)")
print(f"  Animals: {daily_features['tag_short'].nunique()}")
print(f"  Days: {daily_features['date'].nunique()}")
print(f"\nFeature statistics (computed from drinking bouts, not raw scanner sessions):")
print(daily_features[['total_daily_duration', 'visit_count', 'avg_visit_duration', 'max_absence_hours']].describe().round(2))
daily_features.head(10)

Feature matrix: 793 rows (animal-days)
  Animals: 30
  Days: 27

Feature statistics (computed from drinking bouts, not raw scanner sessions):
       total_daily_duration  visit_count  avg_visit_duration  \
count                793.00       793.00              793.00   
mean                 412.87         6.85               70.71   
std                  299.48         5.06               47.55   
min                    0.12         1.00                0.12   
25%                  187.87         3.00               39.34   
50%                  347.35         5.00               61.37   
75%                  557.92         9.00               90.21   
max                 1826.51        33.00              346.43   

       max_absence_hours  
count             792.00  
mean               14.53  
std                 7.61  
min                 1.67  
25%                 9.95  
50%                13.46  
75%                16.77  
max                63.68  


,tag_short,date,total_daily_duration,visit_count,avg_visit_duration,max_absence_sec,max_absence_hours
0,1101,2024-10-11,1333.912,14,95.279429,16189.525,4.497090
1,1101,2024-10-12,699.267,8,87.408375,32042.840,8.900789
2,1101,2024-10-13,777.478,11,70.679818,25827.244,7.174234
3,1101,2024-10-14,481.809,6,80.301500,32939.937,9.149983
4,1101,2024-10-16,492.113,5,98.422600,160458.962,44.571934
5,1101,2024-10-17,710.507,9,78.945222,29078.569,8.077380
6,1101,2024-10-18,1826.510,12,152.209167,30637.358,8.510377
7,1101,2024-10-19,579.937,9,64.437444,37937.202,10.538112
8,1101,2024-10-20,840.465,8,105.058125,34707.380,9.640939
9,1101,2024-10-21,800.113,6,133.352167,26750.687,7.430746


## Step 2: Establish Per-Animal Baselines

In [4]:
# ---- Per-animal baselines (mean and std across all days) ----
# This establishes what "normal" looks like for each individual animal.

feature_cols = ['total_daily_duration', 'visit_count', 'avg_visit_duration', 'max_absence_hours']

baselines = daily_features.groupby('tag_short')[feature_cols].agg(['mean', 'std']).round(2)
baselines.columns = ['_'.join(col) for col in baselines.columns]
baselines = baselines.reset_index()

# Merge health labels
baselines = baselines.merge(
    df_animals[['tag_short', 'health_note', 'is_sick']], 
    on='tag_short', how='left'
)

print("Per-animal baselines (mean ± std across all days):")
print(baselines.to_string(index=False))

Per-animal baselines (mean ± std across all days):
 tag_short  total_daily_duration_mean  total_daily_duration_std  visit_count_mean  visit_count_std  avg_visit_duration_mean  avg_visit_duration_std  max_absence_hours_mean  max_absence_hours_std   health_note  is_sick
      1101                     623.79                    337.88              7.07             2.72                    88.42                   30.44                   12.54                   7.38           NaN        0
      1105                     455.89                    242.89              6.93             4.07                    74.20                   33.49                   16.26                   6.91           NaN        0
      1106                     616.88                    384.45              6.81             4.61                   104.39                   65.39                   12.73                   5.58           NaN        0
      1110                     127.65                     83.74              

In [5]:
# ---- Compare sick animals vs herd averages ----

herd_means = daily_features[feature_cols].mean()
print("Herd-wide daily averages:")
for col in feature_cols:
    unit = 'sec' if 'duration' in col else ('visits' if 'count' in col else 'hours')
    print(f"  {col}: {herd_means[col]:.1f} {unit}")

print("\nSick animals vs herd average:")
sick_tags = df_animals[df_animals['is_sick'] == 1]['tag_short'].tolist()
# Only compare sick animals that have data in our dataset
sick_with_data = [t for t in sick_tags if t in daily_features['tag_short'].values]

for tag in sick_with_data:
    animal_means = daily_features[daily_features['tag_short'] == tag][feature_cols].mean()
    note = df_animals[df_animals['tag_short'] == tag]['health_note'].values[0]
    print(f"\n  Animal {tag} ({note}):")
    for col in feature_cols:
        val = animal_means[col]
        herd_val = herd_means[col]
        pct_diff = ((val - herd_val) / herd_val) * 100
        direction = "above" if pct_diff > 0 else "below"
        print(f"    {col}: {val:.1f} ({abs(pct_diff):.0f}% {direction} herd avg)")

Herd-wide daily averages:
  total_daily_duration: 412.9 sec
  visit_count: 6.9 visits
  avg_visit_duration: 70.7 sec
  max_absence_hours: 14.5 hours

Sick animals vs herd average:

  Animal 1129 (пневмония??):
    total_daily_duration: 409.1 (1% below herd avg)
    visit_count: 6.7 (3% below herd avg)
    avg_visit_duration: 66.5 (6% below herd avg)
    max_absence_hours: 15.5 (7% above herd avg)

  Animal 1149 (тимпания факт):
    total_daily_duration: 294.4 (29% below herd avg)
    visit_count: 3.0 (56% below herd avg)
    avg_visit_duration: 123.0 (74% above herd avg)
    max_absence_hours: 22.6 (55% above herd avg)


## Step 3: Add Health Labels and Export

In [6]:
# ---- Merge health labels into daily features ----

daily_features = daily_features.merge(
    df_animals[['tag_short', 'health_note', 'is_sick']],
    on='tag_short', how='left'
)

# ---- Export ----
output_path = os.path.join(PROCESSED_DIR, 'daily_features.csv')
daily_features.to_csv(output_path, index=False)

print(f"Exported daily_features.csv: {len(daily_features)} rows")
print(f"  Columns: {list(daily_features.columns)}")
print(f"  Sick animal-days: {daily_features['is_sick'].sum()}")
print(f"  Healthy animal-days: {(daily_features['is_sick'] == 0).sum()}")
print(f"\nSaved to: {output_path}")
daily_features.tail(10)

Exported daily_features.csv: 793 rows
  Columns: ['tag_short', 'date', 'total_daily_duration', 'visit_count', 'avg_visit_duration', 'max_absence_sec', 'max_absence_hours', 'health_note', 'is_sick']
  Sick animal-days: 49
  Healthy animal-days: 744

Saved to: /Users/dimvsh/Downloads/Dana Thesis/project/data/processed/daily_features.csv


,tag_short,date,total_daily_duration,visit_count,avg_visit_duration,max_absence_sec,max_absence_hours,health_note,is_sick
783,1150,2024-10-29,329.279,10,32.927900,50118.063,13.921684,NaN,0
784,1150,2024-10-30,368.936,13,28.379692,35820.403,9.950112,NaN,0
785,1150,2024-10-31,538.656,8,67.332000,26883.398,7.467611,NaN,0
786,1150,2024-11-01,360.189,21,17.151857,29004.522,8.056812,NaN,0
787,1150,2024-11-02,91.243,2,45.621500,56862.666,15.795185,NaN,0
788,1150,2024-11-03,936.308,10,93.630800,51389.265,14.274796,NaN,0
789,1150,2024-11-04,589.612,5,117.922400,41066.836,11.407454,NaN,0
790,1150,2024-11-05,325.710,3,108.570000,43668.746,12.130207,NaN,0
791,1150,2024-11-06,163.888,9,18.209778,60554.286,16.820635,NaN,0
792,1150,2024-11-07,384.713,18,21.372944,41934.952,11.648598,NaN,0
